# Exploratory Notebook

## Download ERA 5 input files

The files will be downloaded in two groups. First the geopotential variable can be downloaded for any given date as it will be constant in any other time.

In [13]:
import cdsapi
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd().parent
DOWNLOAD_DIR = REPO_ROOT / "data"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

DATASET = "reanalysis-era5-single-levels"

client = cdsapi.Client()

base_request = {
    "product_type": ["reanalysis"],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [13, -79, -4, -66]
}


def safe_retrieve(dataset: str, request: dict, target_path: Path) -> None:
    """Download from CDS to a temporary .part file before renaming to final target."""
    if target_path.is_file():
        print(f"[SKIP] {target_path.name} already exists.")
        return

    temp_path = target_path.with_suffix(target_path.suffix + ".part")
    print(f"[FETCH] Downloading {target_path.name}...")
    client.retrieve(dataset, request, str(temp_path))
    temp_path.rename(target_path)


# --- Step 1: Time-invariant Surface Geopotential ---
geopotential_path = DOWNLOAD_DIR / "era5_colombia_geopotential.nc"
geopotential_request = base_request | {
    "variable": ["geopotential"],
    "year": ["2025"],
    "month": ["01"],
    "day": ["01"],
    "time": ["00:00"],
}
safe_retrieve(DATASET, geopotential_request, geopotential_path)

2026-11-24 16:10:13,650 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.


[SKIP] era5_colombia_geopotential.nc already exists.


For the temperature and dewpoint variables, we are going to download a month with the two variables, like this:

In [15]:
year = 2025
month = 1
monthly_base_request = base_request | {
    "variable": [
        "2m_dewpoint_temperature",
        "2m_temperature"
    ],
    "year": [f"{year}"],
    "month": [f"{month:02d}"],
    "day": [f"{d:02d}" for d in range(1, 32)],
    "time": [f"{h:02d}:00" for h in range(24)]
}

filename = f"era5_colombia_2m-temp_2m-dew_{year}-{month:02d}.nc"
file_path = DOWNLOAD_DIR / filename

safe_retrieve(DATASET, monthly_base_request, file_path)

[SKIP] era5_colombia_2m-temp_2m-dew_2025-01.nc already exists.


Then we can download all the files in a loop.

In [ ]:
# --- Step 2: Hourly 2m Temperature & Dewpoint (2012–2025) ---
monthly_base_request = base_request | {
    "variable": [
        "2m_dewpoint_temperature",
        "2m_temperature",
    ],
    "day": [f"{d:02d}" for d in range(1, 32)],
    "time": [f"{h:02d}:00" for h in range(24)],
}

for year in range(2012, 2026):
    for month in range(1, 13):
        filename = f"era5_colombia_2m-temp_2m-dew_{year}-{month:02d}.nc"
        file_path = DOWNLOAD_DIR / filename

        request = monthly_base_request | {
            "year": [f"{year}"],
            "month": [f"{month:02d}"],
        }
        safe_retrieve(DATASET, request, file_path)

[FETCH] Downloading era5_colombia_2m-temp_2m-dew_2012-01.nc...


2026-11-24 16:26:06,142 INFO Request ID is 1c51ea2f-d925-442e-928d-e44340ad4634
2026-11-24 16:26:06,351 INFO status has been updated to accepted
2026-11-24 16:26:41,571 INFO status has been updated to running
2026-11-24 16:29:05,477 INFO status has been updated to successful
                                                                               

[FETCH] Downloading era5_colombia_2m-temp_2m-dew_2012-02.nc...


2026-11-24 16:29:11,234 INFO Request ID is 578399fa-3c8f-416a-9a58-0c9229a8fe8c
2026-11-24 16:29:11,462 INFO status has been updated to accepted
2026-11-24 16:29:25,728 INFO status has been updated to running
2026-11-24 16:31:07,049 INFO status has been updated to successful
                                                                               

[FETCH] Downloading era5_colombia_2m-temp_2m-dew_2012-03.nc...


2026-11-24 16:31:11,206 INFO Request ID is e5c2d271-7583-4918-b6ac-d0d44cf85f4c
2026-11-24 16:31:11,425 INFO status has been updated to accepted
2026-11-24 16:31:25,776 INFO status has been updated to running
2026-11-24 16:33:07,031 INFO status has been updated to successful
                                                                               

... (238 outputs left out)

[FETCH] Downloading era5_colombia_2m-temp_2m-dew_2022-03.nc...


2026-11-24 22:33:36,966 INFO Request ID is a3b342a7-e497-42cf-9c1f-432fa0e14165
2026-11-24 22:33:37,183 INFO status has been updated to accepted
2026-11-24 22:33:59,815 INFO status has been updated to running
2026-11-24 22:35:34,746 INFO status has been updated to accepted
2026-11-24 22:38:03,848 INFO status has been updated to running
2026-11-24 22:40:07,156 INFO status has been updated to successful
                                                                               

[FETCH] Downloading era5_colombia_2m-temp_2m-dew_2022-04.nc...


2026-11-24 22:40:11,200 INFO Request ID is a2f4dcb1-f98c-45b2-aaa8-178b01eba008
2026-11-24 22:40:11,423 INFO status has been updated to accepted
2026-11-24 22:40:33,650 INFO status has been updated to running
